# 20. Colab 27B 환경 준비와 사전 점검

담당: 준혁 한 사람만 Colab 런타임을 조작합니다.
목표: 자원을 기록하고, 데이터와 가중치를 준비하고, 버전을 고정하고, 실제 측정값으로 시간과
compute unit 예산을 계산하는 것입니다.

이 노트북은 준비와 측정만 담당합니다. 오래 걸리는 실제 실행은 `21_colab_27b_run.ipynb` 에서
분리된 job으로 띄웁니다.

## 통과 기준(PASS)
1. GPU, VRAM, RAM, 디스크, Drive 여유, 잔여 CU를 모두 기록했습니다.
2. 데이터가 Drive가 아니라 런타임 로컬 디스크에 풀려 있습니다.
3. `RUN_ROOT/env_lock.txt` 에 설치된 버전이 고정되어 있습니다.
4. 27B snapshot을 revision과 함께 받았습니다.
5. preflight가 20 optimizer step과 50문항 추론을 통과했습니다.
6. 예산 계산 결과가 잔여 CU의 80퍼센트 안에 들어옵니다.

## 넘겨줄 내용
마지막 셀의 예산 블록을 팀 채널에 공유합니다. 여기에서 27B가 예산을 통과하지 못하면 9B 경로로
전환해야 하므로, 판단을 미루지 말고 바로 알려야 합니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

In [12]:
# ===== 파라미터 =====
PROJECT_ROOT = "/content/ssafy-16-2-ai"          # 코드 저장소를 풀어 둘 위치
REPO_URL = "https://github.com/bnbong/Astralopithecus.git"   # 비공개 저장소입니다.
REPO_BRANCH = "main"
CODE_ZIP = ""                                     # 토큰을 쓸 수 없을 때 사용할 Drive의 코드 zip 경로
DRIVE_MOUNT = "/content/drive"
DRIVE_DIR = "/content/drive/MyDrive/vqa"          # 영속 산출물을 둘 Drive 폴더
ARCHIVE_PATH = "/content/drive/MyDrive/ssafy-16-2-ai.zip"   # 암호가 걸린 데이터 압축 파일
ARCHIVE_PASSWORD = ""                             # 비워 두면 실행할 때 입력받습니다.

LOCAL_SCRATCH = "/content/scratch"
DATA_ROOT = "/content/data"                       # 압축을 풀 로컬 디스크 위치
RUN_ROOT = "/content/drive/MyDrive/vqa/runs"      # checkpoint가 남을 영속 위치
MODEL_ROOT = ""                                   # 아래 셀에서 정해집니다.

MODEL_ID = "Qwen/Qwen3.5-27B"
MODEL_REVISION = "main"                           # 실제 commit 해시로 고정하는 편이 좋습니다.

# 수동으로 확인해서 적는 값입니다. Colab에는 잔여 CU를 조회하는 API가 없습니다.
CU_REMAINING = 600.0        # 계정 화면에서 본 잔여 compute unit
CU_PER_HOUR = 0.0           # 계정 화면에서 본 현재 GPU의 시간당 소모량. 모르면 0으로 두십시오.
CU_EXPIRY = "미확인"         # 만료일

N_TEST_ITEMS = 6714
N_VAL_ITEMS = 500           # 고정 split의 val 문항 수. split 파일이 있으면 아래에서 덮어씁니다.
N_TRAIN_SAMPLES = 6214      # 고정 split의 train 표본 수. split 파일이 있으면 아래에서 덮어씁니다.
MICRO_BATCH = 1
GRAD_ACCUM = 8

## 1. Drive 연결

런타임 조작자는 준혁 한 사람입니다. 다른 팀원이 같은 노트북을 열어도 실행 중인 가상 머신은
공유되지 않으므로, 진행 상황은 Drive에 저장된 로그로 확인해야 합니다.

In [13]:
from google.colab import drive
drive.mount(DRIVE_MOUNT)
print("Drive를 연결했습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive를 연결했습니다.


## 1-1. 코드 가져오기

새로 만들어진 Colab 가상 머신에는 저장소가 들어 있지 않습니다. 부트스트랩 셀이 `PROJECT_ROOT` 로
작업 디렉터리를 옮기기 때문에, 그보다 먼저 이 셀로 코드를 내려받아야 합니다.

저장소가 비공개이므로 인증 수단이 필요하며, 다음 두 가지 중 하나를 준비하면 됩니다.

1. GitHub의 Settings에서 이 저장소에만 Contents 읽기 권한을 준 fine-grained personal access
   token을 만듭니다. 그다음 Colab 화면 왼쪽의 열쇠 아이콘 패널을 열고, 이름을 `GITHUB_TOKEN` 으로
   해서 토큰 값을 등록하고 현재 노트북의 접근을 허용합니다. 토큰 값은 화면과 로그에 출력되지
   않으며, 복제가 끝나는 즉시 remote 주소에서 제거하므로 `.git/config` 에도 남지 않습니다.
2. 토큰을 쓰기 어려운 상황이라면 코드 전체를 zip으로 묶어 Drive에 올리고, 파라미터 셀의
   `CODE_ZIP` 에 그 경로를 적습니다. 압축 안에 최상위 폴더가 하나만 있어도 알아서 펼칩니다.

이미 코드를 받아 둔 상태라면 `git pull --ff-only` 로 갱신만 합니다. 갱신이 실패하더라도 멈추지
않고 기존 코드를 그대로 사용합니다.

In [14]:
# --- 코드 가져오기 (부트스트랩보다 먼저 실행합니다. BOOT의 헬퍼를 쓰지 않습니다.) ---
import os, re, shutil, subprocess, zipfile
from pathlib import Path

_code_root = Path(PROJECT_ROOT).expanduser()


def scrub_secret(text, secret=None):
    """출력에 토큰이 섞여 나가지 않도록 가립니다."""
    if not text:
        return ""
    if secret:
        text = text.replace(secret, "***")
    return re.sub(r"(https://)[^/@\s]*@", r"\1***@", text)


def _git(args, secret=None, check=True):
    """git 명령을 조용히 실행하고, 출력에서 토큰을 지운 뒤 돌려줍니다."""
    proc = subprocess.run(
        ["git"] + [str(a) for a in args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    )
    out = scrub_secret(proc.stdout, secret)
    if check and proc.returncode != 0:
        raise RuntimeError("git 명령이 실패했습니다 (rc=%d).\n%s" % (proc.returncode, out.strip()))
    return proc.returncode, out


def _read_github_token():
    """Colab 비밀 저장소에서 토큰을 읽습니다. 없으면 None을 돌려줍니다."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or None
    except Exception:
        # SecretNotFoundError, NotebookAccessError, ImportError를 모두 여기에서 받습니다.
        return None


def _flatten_single_top_dir(target):
    """압축 안에 최상위 폴더가 하나뿐이면 그 안의 내용을 target 바로 아래로 올립니다."""
    entries = [p for p in target.iterdir() if p.name != "__MACOSX"]
    if len(entries) == 1 and entries[0].is_dir():
        inner = entries[0]
        for item in list(inner.iterdir()):
            shutil.move(str(item), str(target / item.name))
        inner.rmdir()
        return True
    return False


CODE_VERSION = None

if (_code_root / ".git").exists():
    rc, out = _git(["-C", str(_code_root), "pull", "--ff-only"], check=False)
    if rc == 0:
        print("코드를 최신 상태로 갱신했습니다.")
    else:
        print("git pull이 실패했습니다. 이미 받아 둔 코드를 그대로 사용하고 계속 진행합니다.")
        print(out.strip()[:600])
    _, _head = _git(["-C", str(_code_root), "rev-parse", "HEAD"], check=False)
    CODE_VERSION = _head.strip()
else:
    _token = _read_github_token()
    if _token:
        print("Colab 비밀 GITHUB_TOKEN을 찾았습니다. 저장소를 복제합니다.")
        _code_root.parent.mkdir(parents=True, exist_ok=True)
        _auth_url = REPO_URL.replace("https://", "https://x-access-token:%s@" % _token, 1)
        _rc, _out = _git(["clone", "--depth", "1", "--branch", REPO_BRANCH,
                          _auth_url, str(_code_root)], secret=_token, check=False)
        if _rc != 0:
            raise RuntimeError("저장소 복제에 실패했습니다. 토큰 권한과 저장소 주소를 확인하십시오.\n"
                               + _out.strip()[:600])
        # 토큰이 .git/config에 남지 않도록 곧바로 원래 주소로 되돌립니다.
        _git(["-C", str(_code_root), "remote", "set-url", "origin", REPO_URL], secret=_token)
        _, _head = _git(["-C", str(_code_root), "rev-parse", "HEAD"], check=False)
        CODE_VERSION = _head.strip()
        del _auth_url, _token
    elif CODE_ZIP and Path(CODE_ZIP).expanduser().exists():
        _zip_path = Path(CODE_ZIP).expanduser()
        print("토큰이 없어 Drive의 코드 압축 파일을 사용합니다:", _zip_path)
        _code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(_zip_path) as _zf:
            _zf.extractall(_code_root)
        if _flatten_single_top_dir(_code_root):
            print("압축 안의 최상위 폴더 하나를 펼쳐서 %s 바로 아래로 옮겼습니다." % _code_root)
        CODE_VERSION = "zip:" + _zip_path.name
    else:
        raise RuntimeError(
            "코드를 가져올 방법이 없습니다. 이 저장소는 비공개이므로 다음 두 가지 중 하나를 준비해야 합니다.\n"
            "1) GitHub에서 이 저장소에만 Contents 읽기 권한을 준 fine-grained personal access token을 만든 다음, "
            "Colab 왼쪽의 열쇠 아이콘 패널에서 이름을 GITHUB_TOKEN으로 등록하고 현재 노트북의 접근을 허용하십시오.\n"
            "2) 코드 전체를 zip으로 묶어 Drive에 올린 다음, 파라미터 셀의 CODE_ZIP에 그 경로를 적으십시오."
        )

if not (_code_root / "vqa").is_dir():
    print("주의: %s 아래에 vqa 패키지가 보이지 않습니다. 압축 파일의 구조를 확인하십시오." % _code_root)

print("코드 위치 :", _code_root)
print("코드 버전 :", CODE_VERSION)

git pull이 실패했습니다. 이미 받아 둔 코드를 그대로 사용하고 계속 진행합니다.
fatal: could not read Username for 'https://github.com': No such device or address
코드 위치 : /content/ssafy-16-2-ai
코드 버전 : deb5bf4cf70c5f417c3be15002bd3cf5a5819fca


In [15]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다."""
    return run([PY, str(Path("scripts") / name)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

PROJECT_ROOT      : /content/ssafy-16-2-ai
VQA_DATA_ROOT     : /content/data
VQA_RUN_ROOT      : /content/drive/MyDrive/vqa/runs
VQA_LOCAL_SCRATCH : /content/scratch
VQA_MODEL_ROOT    : (미설정)


In [16]:
# --- 어떤 코드로 실행했는지 남깁니다 ---
_code_version_path = RUN_ROOT_P / "code_version.txt"
with _code_version_path.open("a", encoding="utf-8") as _fh:
    _fh.write("%s\t%s\n" % (time.strftime("%Y-%m-%d %H:%M:%S"),
                            CODE_VERSION or "(알 수 없음)"))
print("코드 버전을 기록했습니다: %s -> %s" % (_code_version_path, CODE_VERSION))


# --- 고정 split이 있으면 표본 수를 실제 값으로 맞춥니다 ---
_split_path = Path("artifacts/data/split_v1.json")
if _split_path.exists():
    _split = json.loads(_split_path.read_text(encoding="utf-8"))
    N_TRAIN_SAMPLES = len(_split["train_ids"])
    N_VAL_ITEMS = len(_split["val_ids"])
    print("고정 split을 읽어서 표본 수를 갱신했습니다: train %d / val %d"
          % (N_TRAIN_SAMPLES, N_VAL_ITEMS))
else:
    print("%s 이 아직 없어서 파라미터 셀의 기본값을 그대로 씁니다: train %d / val %d"
          % (_split_path, N_TRAIN_SAMPLES, N_VAL_ITEMS))

코드 버전을 기록했습니다: /content/drive/MyDrive/vqa/runs/code_version.txt -> deb5bf4cf70c5f417c3be15002bd3cf5a5819fca
고정 split을 읽어서 표본 수를 갱신했습니다: train 6214 / val 500


## 2. 자원 기록

GPU 종류와 VRAM, CPU RAM, 로컬 디스크 여유, Drive 여유를 한 번에 기록합니다. 이 값으로 아래에서
bf16과 nf4 중 어느 설정을 쓸지 결정합니다.

In [17]:
import shutil

RESOURCES = {}

smi = run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
           "--format=csv,noheader"], check=False, quiet=True)
if smi["rc"] == 0 and smi["output"].strip():
    first = smi["output"].strip().splitlines()[0]
    parts = [x.strip() for x in first.split(",")]
    RESOURCES["gpu_name"] = parts[0]
    RESOURCES["vram_mib"] = int(parts[1].split()[0])
    RESOURCES["vram_gb"] = round(RESOURCES["vram_mib"] / 1024.0, 1)
    RESOURCES["driver"] = parts[2] if len(parts) > 2 else "?"
else:
    print("nvidia-smi를 사용할 수 없습니다. GPU 런타임인지 확인하십시오.")
    RESOURCES["gpu_name"] = None
    RESOURCES["vram_gb"] = 0.0

try:
    meminfo = Path("/proc/meminfo").read_text()
    total_kb = int([l for l in meminfo.splitlines() if l.startswith("MemTotal")][0].split()[1])
    RESOURCES["ram_gb"] = round(total_kb / 1024.0 / 1024.0, 1)
except Exception as exc:
    RESOURCES["ram_gb"] = None
    print("RAM 정보를 읽지 못했습니다:", exc)

for label, path in (("local_disk", "/content"), ("drive", DRIVE_DIR)):
    try:
        Path(path).mkdir(parents=True, exist_ok=True)
        usage = shutil.disk_usage(path)
        RESOURCES[label + "_free_gb"] = round(usage.free / 1024 ** 3, 1)
        RESOURCES[label + "_total_gb"] = round(usage.total / 1024 ** 3, 1)
    except Exception as exc:
        print("%s 용량을 확인하지 못했습니다: %s" % (label, exc))

for k, v in RESOURCES.items():
    print("%-18s: %s" % (k, v))

$ nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
[rc=0] 0.0초
gpu_name          : NVIDIA A100-SXM4-40GB
vram_mib          : 40960
vram_gb           : 40.0
driver            : 580.82.07
ram_gb            : 83.5
local_disk_free_gb: 188.3
local_disk_total_gb: 235.7
drive_free_gb     : 178.9
drive_total_gb    : 235.7


## 3. 잔여 compute unit 기록

Colab은 잔여 CU를 조회하는 API를 제공하지 않습니다. 화면 오른쪽 위의 자원 표시나 구독 화면에서
직접 확인한 값을 파라미터 셀에 적고, 아래 셀로 기록을 남기십시오. 시간당 소모량을 모르면
30분 정도 실행한 뒤 줄어든 양을 보고 역산합니다.

In [18]:
if CU_PER_HOUR <= 0:
    print("시간당 CU 소모량이 아직 입력되지 않았습니다.")
    print("계산용 가정값 7.5 CU/h를 임시로 사용하되, 실제 값을 확인해서 파라미터 셀을 고치십시오.")
    CU_RATE = 7.5
    CU_RATE_SOURCE = "가정값"
else:
    CU_RATE = CU_PER_HOUR
    CU_RATE_SOURCE = "계정 화면에서 확인"

usable_cu = CU_REMAINING * 0.8
print("잔여 CU        : %.1f (%s 만료)" % (CU_REMAINING, CU_EXPIRY))
print("시간당 소모     : %.2f CU/h (%s)" % (CU_RATE, CU_RATE_SOURCE))
print("20%% 예비를 뺀 가용: %.1f CU = 약 %.1f GPU시간" % (usable_cu, usable_cu / CU_RATE))

RESOURCES["cu_remaining"] = CU_REMAINING
RESOURCES["cu_per_hour"] = CU_RATE
RESOURCES["cu_rate_source"] = CU_RATE_SOURCE
RESOURCES["cu_expiry"] = CU_EXPIRY

시간당 CU 소모량이 아직 입력되지 않았습니다.
계산용 가정값 7.5 CU/h를 임시로 사용하되, 실제 값을 확인해서 파라미터 셀을 고치십시오.
잔여 CU        : 600.0 (미확인 만료)
시간당 소모     : 7.50 CU/h (가정값)
20% 예비를 뺀 가용: 480.0 CU = 약 64.0 GPU시간


## 4. 데이터를 로컬 디스크로 옮기기

Drive를 그대로 읽으면 작은 파일을 반복해서 읽을 때 매우 느립니다. 압축 파일을 런타임 로컬
디스크에 풀어 놓고, 학습과 추론은 로컬 경로만 사용합니다. 이미 풀려 있으면 건너뜁니다.

압축 파일에 암호가 걸려 있지 않다면 파라미터 셀의 `ARCHIVE_PASSWORD` 에 `none` 이라고 적으십시오.
그러면 입력을 받지 않고 그대로 풉니다. 압축 안에 최상위 폴더가 하나만 있는 경우에는 그 폴더를
데이터 위치로 삼습니다.

In [19]:
# --- 데이터를 로컬 디스크에 준비합니다 ---
import getpass, shutil

data_root = Path(DATA_ROOT)


def locate_data_root(base):
    """압축 안에 최상위 폴더가 하나 있었다면 그 하위 폴더를 데이터 위치로 삼습니다."""
    if (base / "train.csv").exists():
        return base
    if base.is_dir():
        subs = [p for p in sorted(base.iterdir())
                if p.is_dir() and p.name != "__MACOSX" and (p / "train.csv").exists()]
        if len(subs) == 1:
            return subs[0]
    return base


def ensure_7z():
    """7z 명령이 없으면 조용히 설치합니다."""
    if shutil.which("7z"):
        return
    print("7z 명령이 없어 p7zip-full을 설치합니다. 30초 정도 걸립니다.")
    run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=False, quiet=True)
    if not shutil.which("7z"):
        raise RuntimeError("p7zip-full 설치에 실패했습니다. 런타임의 네트워크 상태를 확인하십시오.")


data_root.mkdir(parents=True, exist_ok=True)
found_root = locate_data_root(data_root)

if (found_root / "train.csv").exists() and (found_root / "train").is_dir():
    print("이미 데이터가 준비되어 있습니다:", found_root)
else:
    ensure_7z()
    if str(ARCHIVE_PASSWORD).strip().lower() == "none":
        pwd = None
        print("ARCHIVE_PASSWORD가 none이므로 암호가 없는 압축 파일로 처리합니다.")
    else:
        pwd = ARCHIVE_PASSWORD or getpass.getpass("압축 파일 비밀번호를 입력하십시오: ")
    cmd = ["7z", "x", "-y"]
    if pwd:
        cmd.append("-p" + pwd)
    cmd += ["-o" + str(data_root), ARCHIVE_PATH]
    print("$ 7z x -y %s -o%s %s" % ("-p***" if pwd else "", data_root, ARCHIVE_PATH))
    run(cmd, echo=False)
    found_root = locate_data_root(data_root)

if found_root != data_root:
    print("압축 안의 하위 폴더를 데이터 위치로 사용합니다:", found_root)

data_root = found_root
DATA_ROOT = str(data_root)
DATA_ROOT_P = data_root
os.environ["VQA_DATA_ROOT"] = str(data_root)


for name in ("train.csv", "test.csv", "dev.csv", "sample_submission.csv"):
    p = data_root / name
    print("%-24s %s" % (name, "있음" if p.exists() else "없음"))
for name in ("train", "test", "dev"):
    p = data_root / name
    n = len(list(p.glob("*.jpg"))) if p.is_dir() else 0
    print("%-24s 이미지 %d장" % (name + "/", n))

print("VQA_DATA_ROOT:", os.environ["VQA_DATA_ROOT"])

$ 7z x -y -p*** -o/content/data /content/drive/MyDrive/ssafy-16-2-ai.zip

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:12 OPEN_MAX:1048576

Scanning the drive for archives:
1 file, 1799380978 bytes (1717 MiB)

Extracting archive: /content/drive/MyDrive/ssafy-16-2-ai.zip
--
Path = /content/drive/MyDrive/ssafy-16-2-ai.zip
Type = zip
Physical Size = 1799380978

Everything is Ok

Folders: 3
Files: 16118
Size:       1801552403
Compressed: 1799380978
[rc=0] 31.7초
train.csv                있음
test.csv                 있음
dev.csv                  있음
sample_submission.csv    있음
train/                   이미지 6714장
test/                    이미지 6714장
dev/                     이미지 2683장
VQA_DATA_ROOT: /content/data


## 5. 라이브러리 설치와 버전 고정

설치가 끝나면 곧바로 `pip freeze` 결과를 Drive에 저장합니다. 가상 머신이 재활용되어도 같은
환경을 다시 만들 수 있어야 하기 때문입니다.

In [20]:
REQ = "requirements/colab.txt"
if Path(REQ).exists():
    run([PY, "-m", "pip", "install", "-q", "-r", REQ])
else:
    print(REQ + " 이 없습니다. 코어 담당자의 커밋을 기다렸다가 다시 실행하십시오.")

$ /usr/bin/python3 -m pip install -q -r requirements/colab.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 30.9 MB/s eta 0:00:00
[rc=0] 5.0초


In [21]:
freeze = run([PY, "-m", "pip", "freeze"], quiet=True)
lock_path = RUN_ROOT_P / "env_lock.txt"
lock_path.write_text(freeze["output"], encoding="utf-8")
print("버전을 고정했습니다:", lock_path)
print("총 %d개 패키지" % len(freeze["output"].splitlines()))
for line in freeze["output"].splitlines():
    if line.lower().startswith(("torch", "transformers", "peft", "accelerate", "bitsandbytes")):
        print("  " + line)

$ /usr/bin/python3 -m pip freeze
[rc=0] 1.4초
버전을 고정했습니다: /content/drive/MyDrive/vqa/runs/env_lock.txt
총 706개 패키지
  accelerate==1.14.0
  bitsandbytes==0.50.2
  peft==0.20.0
  torch==2.11.0+cu128
  torchao==0.10.0
  torchaudio==2.11.0+cu128
  torchcodec==0.11.0+cu128
  torchdata==0.11.0
  torchsummary==1.5.1
  torchtune==0.6.1
  torchvision==0.26.0+cu128
  transformers==5.16.1


## 6. 27B 가중치 받기

Drive 여유 공간이 충분하면 Drive의 캐시에 받아 둡니다. 가상 머신이 바뀌어도 다시 받지 않아도
되기 때문입니다. 여유가 부족하면 로컬 디스크에 받습니다. 이 경우 런타임이 재활용될 때마다
다시 받아야 하며, 회선 상태에 따라 수십 분이 걸릴 수 있습니다.

revision을 `main` 으로 두지 말고 실제 commit 해시로 고정하는 편이 재현에 유리합니다.

In [22]:
NEEDED_GB = 60.0   # 27B 원본 가중치에 필요한 대략적인 여유 공간
drive_free = RESOURCES.get("drive_free_gb", 0.0)
local_free = RESOURCES.get("local_disk_free_gb", 0.0)

if drive_free >= NEEDED_GB:
    CACHE_DIR = str(Path(DRIVE_DIR) / "hf_cache")
    CACHE_NOTE = "Drive 캐시에 받았습니다. 런타임이 바뀌어도 다시 받지 않아도 됩니다."
else:
    CACHE_DIR = "/content/hf_cache"
    CACHE_NOTE = ("Drive 여유가 %.1fGB뿐이라 로컬 디스크에 받았습니다. "
                  "가상 머신이 재활용되면 다시 받아야 하며 수십 분이 걸릴 수 있습니다." % drive_free)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = CACHE_DIR
os.environ["VQA_MODEL_ROOT"] = CACHE_DIR
print("캐시 위치:", CACHE_DIR)
print(CACHE_NOTE)
print("로컬 디스크 여유: %.1fGB / Drive 여유: %.1fGB" % (local_free, drive_free))

캐시 위치: /content/drive/MyDrive/vqa/hf_cache
Drive 캐시에 받았습니다. 런타임이 바뀌어도 다시 받지 않아도 됩니다.
로컬 디스크 여유: 188.3GB / Drive 여유: 178.9GB


In [23]:
os.environ["VQA_MODEL_ID"] = MODEL_ID
os.environ["VQA_MODEL_REV"] = MODEL_REVISION
print("모델 식별자와 revision을 환경 변수에 넣었습니다.")

모델 식별자와 revision을 환경 변수에 넣었습니다.


In [24]:
hf_bin = shutil.which("hf") or shutil.which("huggingface-cli")
if hf_bin:
    dl = run([hf_bin, "download", MODEL_ID, "--revision", MODEL_REVISION,
              "--cache-dir", CACHE_DIR], check=False)
else:
    print("hf 명령을 찾지 못해 python 모듈로 받습니다.")
    dl = run([PY, "-c",
              "import os, huggingface_hub as h;"
              "p=h.snapshot_download(os.environ['VQA_MODEL_ID'],"
              " revision=os.environ['VQA_MODEL_REV'], cache_dir=os.environ['HF_HOME']);"
              "print('snapshot:', p)"],
             check=False)

MODEL_SNAPSHOT_NOTE = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "cache_dir": CACHE_DIR,
    "download_rc": dl["rc"],
    "download_seconds": round(dl["seconds"], 1),
}
(RUN_ROOT_P / "model_snapshot.json").write_text(
    json.dumps(MODEL_SNAPSHOT_NOTE, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(MODEL_SNAPSHOT_NOTE, ensure_ascii=False, indent=2))
if dl["rc"] != 0:
    print()
    print("다운로드가 실패했습니다. 네트워크 상태와 남은 디스크 용량을 확인한 뒤 이 셀을 다시 실행하십시오.")
    print("이미 받은 부분은 캐시에 남아 있으므로 처음부터 다시 받지 않습니다.")

$ /usr/local/bin/hf download Qwen/Qwen3.5-27B --revision main --cache-dir /content/drive/MyDrive/vqa/hf_cache
Hint: A new version of huggingface_hub (1.32.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 24 files: 100%|██████████| 24/24 [02:55<00:00,  7.29s/it]
✓ Downloaded
  path: /content/drive/MyDrive/vqa/hf_cache/models--Qwen--Qwen3.5-27B/snapshots/fc05daec18b0a78c049392ed2e771dde82bdf654
[rc=0] 177.2초
{
  "model_id": "Qwen/Qwen3.5-27B",
  "revision": "main",
  "cache_dir": "/content/drive/MyDrive/vqa/hf_cache",
  "download_rc": 0,
  "download_seconds": 177.2
}


## 7. VRAM으로 config 고르기

계획서 2절의 기준을 그대로 적용합니다.

| 감지된 VRAM | 사용할 설정 |
|---|---|
| 70GB 이상 | `colab_qwen35_27b_bf16.yaml` |
| 35GB 이상 70GB 미만 | `colab_qwen35_27b_nf4.yaml` |
| 35GB 미만 | 27B를 포기하고 9B 경로로 전환합니다. |

세 번째 경우에는 아래 셀이 오류를 내면서 멈춥니다. 그때는 팀에 알리고 실습실 9B 경로에
인력을 더 투입해야 합니다.

In [25]:
vram = RESOURCES.get("vram_gb", 0.0)
if vram >= 70:
    CONFIG = "configs/colab_qwen35_27b_bf16.yaml"
    PRECISION = "bf16"
elif vram >= 35:
    CONFIG = "configs/colab_qwen35_27b_nf4.yaml"
    PRECISION = "nf4"
else:
    raise RuntimeError(
        "감지된 VRAM이 %.1fGB입니다. 27B 경로를 포기하고 9B 경로로 전환해야 합니다. "
        "팀 채널에 즉시 알리고 docs/README_team.md의 대체 경로를 따르십시오." % vram
    )

print("감지된 VRAM %.1fGB -> %s (%s)" % (vram, CONFIG, PRECISION))
print("config 존재:", Path(CONFIG).exists())

# 21번 노트북이 같은 config를 그대로 쓰도록 선택 결과를 남깁니다.
COLAB_CHOICE = {
    "config": CONFIG,
    "precision": PRECISION,
    "vram_gb": vram,
    "gpu_name": RESOURCES.get("gpu_name"),
    "code_version": CODE_VERSION,
    "decided_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(RUN_ROOT_P / "colab_choice.json").write_text(
    json.dumps(COLAB_CHOICE, ensure_ascii=False, indent=2), encoding="utf-8")
print("선택 결과를 저장했습니다:", RUN_ROOT_P / "colab_choice.json")

감지된 VRAM 40.0GB -> configs/colab_qwen35_27b_nf4.yaml (nf4)
config 존재: True
선택 결과를 저장했습니다: /content/drive/MyDrive/vqa/runs/colab_choice.json


## 8. preflight

20 optimizer step과 50문항 추론을 측정합니다. 누적이 8이면 20 optimizer step은 160
microbatch입니다. preflight는 먼저 가장 긴 표본 몇 개로 OOM을 보고, 그 다음 무작위 표본
(최대 `--max-timing-microbatches`개)으로 대표 속도를 잽니다. 가장 긴 표본만으로 잰 시간은
상한이라 예산에 쓰면 안 됩니다.

출력에서 `sec_per_microbatch_mean`(초/microbatch)과 `sec_per_optimizer_step`(초/optimizer
step)을 구분해서 확인하십시오. 예산 계산에 들어가는 값은 뒤쪽입니다.

In [26]:
OVERRIDES = []
frozen_path = Path("artifacts/input_config_frozen.json")
if frozen_path.exists():
    frozen = json.loads(frozen_path.read_text(encoding="utf-8"))
    OVERRIDES += ["--set", "prompt_id=%s" % frozen["prompt_id"]]
    mp = frozen.get("max_pixels", {}).get("colab_qwen35_27b")
    if mp:
        OVERRIDES += ["--set", "max_pixels=%d" % mp]
    print("창혁이 동결한 입력 설정을 적용합니다:", OVERRIDES)

pf = vqa("preflight",
         "--config", CONFIG,
         "--n-train-steps", 20,
         "--n-infer", 50,
         "--max-timing-microbatches", 64,
         *OVERRIDES,
         check=False)

$ /usr/bin/python3 -m vqa.preflight --config configs/colab_qwen35_27b_nf4.yaml --n-train-steps 20 --n-infer 50 --max-timing-microbatches 64
[PASS] environment — NVIDIA A100-SXM4-40GB 39.49GB, torch 2.11.0+cu128
[PASS] bitsandbytes_4bit — 4-bit forward+backward 정상
[PASS] data — 6714행, 라벨 6714건
[PASS] processor — Qwen3VLProcessor (14.7s)
[PASS] choice_single_token — a/b/c/d 모두 단일 토큰
[PASS] chat_template_tail — thinking 비활성 확인
[PASS] choice_token_after_tail — scoring token ids=[64, 65, 66, 67]
[PASS] label_mask — 표본당 정답 토큰 1개만 감독
[PASS] train_infer_prefix_match — 4개 표본 모두 추론 prefix와 토큰 단위로 일치

Fetching 11 files: 100%|██████████| 11/11 [02:50<00:00, 15.52s/it]

Loading weights: 100%|██████████| 1184/1184 [03:17<00:00,  6.00it/s]
[PASS] model_load — AutoModelForMultimodalLM, trainable=116,727,808 / 15,070,229,744
[PASS] lora_targets — language 모듈 496개, leaf ['down_proj', 'gate_proj', 'in_proj_a', 'in_proj_b', 'in_proj_qkv', 'in_proj_z', 'k_proj', 'o_proj', 'out_proj', 'q_proj', 'up_proj', '

In [27]:
# preflight 결과 JSON에서 측정값을 읽습니다.
pf_dir = RUN_ROOT_P / "preflight"
PF = {}
if pf_dir.is_dir():
    files = sorted(pf_dir.glob("*.json"))
    if files:
        PF = json.loads(files[-1].read_text(encoding="utf-8"))
        print("preflight 파일:", files[-1].name)

def pick(node, *keys):
    """preflight 결과는 검사별로 중첩되어 있으므로 키를 재귀로 찾습니다."""
    if isinstance(node, dict):
        for k in keys:
            if k in node:
                return node[k]
        for v in node.values():
            got = pick(v, *keys)
            if got is not None:
                return got
    elif isinstance(node, list):
        for v in node:
            got = pick(v, *keys)
            if got is not None:
                return got
    return None


SEC_PER_ITEM = pick(PF, "seconds_per_item", "sec_per_item")
PEAK_VRAM = pick(PF, "peak_vram_gb", "peak_memory_gb")

# 단위 주의: preflight의 forward+backward는 microbatch 1개(표본 MICRO_BATCH개)를 잽니다.
# optimizer step 1개는 microbatch GRAD_ACCUM개이므로, 예산에는 초/optimizer step을 써야 합니다.
SEC_PER_OPT_STEP = pick(PF, "sec_per_optimizer_step")
SEC_PER_MICRO = pick(PF, "sec_per_microbatch_mean", "warm_seconds_mean")

if SEC_PER_OPT_STEP is not None:
    SEC_PER_STEP = SEC_PER_OPT_STEP
elif SEC_PER_MICRO is not None:
    SEC_PER_STEP = SEC_PER_MICRO * GRAD_ACCUM
    print("preflight가 microbatch 단위 값만 남겼습니다. 누적 %d를 곱해 optimizer step으로 바꿉니다."
          % GRAD_ACCUM)
else:
    # 참고 실행(A100-80GB, 27B bf16)의 3.37초는 microbatch 1개 기준입니다. 여기서만 누적을 곱합니다.
    SEC_PER_MICRO = 3.37
    SEC_PER_STEP = SEC_PER_MICRO * GRAD_ACCUM
    print("학습 속도를 읽지 못해 참고 수치 3.37초/microbatch를 사용합니다. 실제 측정값으로 바꾸십시오.")

if SEC_PER_ITEM is None:
    SEC_PER_ITEM = 0.55
    print("추론 속도를 읽지 못해 참고 수치 0.55초/문항을 사용합니다. 실제 측정값으로 바꾸십시오.")

print("문항당 추론    : %.3f초/문항" % SEC_PER_ITEM)
if SEC_PER_MICRO is not None:
    print("microbatch     : %.2f초/microbatch" % SEC_PER_MICRO)
print("optimizer step : %.2f초/optimizer step (microbatch %d개)" % (SEC_PER_STEP, GRAD_ACCUM))
print("peak VRAM      : %s GB" % PEAK_VRAM)

preflight 파일: 20260921T050017Z.json
문항당 추론    : 1.061초/문항
microbatch     : 7.95초/microbatch
optimizer step : 63.57초/optimizer step (microbatch 8개)
peak VRAM      : 27.99515676498413 GB


## 9. 시간과 CU 예산

실측값으로 base validation, base 전체 test, 학습, FT validation, FT 전체 test를 모두 더합니다.
여기에 20퍼센트 예비를 붙인 값이 잔여 CU 안에 들어와야 27B 경로를 유지할 수 있습니다.

단위를 반드시 구분하십시오. preflight가 재는 것은 **초/microbatch**이고, 학습 시간 계산에
들어가는 것은 **초/optimizer step**입니다. 둘은 누적(GRAD_ACCUM, 기본 8)배만큼 차이가 납니다. 앞 셀에서
`sec_per_optimizer_step`을 그대로 읽고, 그 키가 없을 때만 microbatch 값에 누적을 곱합니다.

예산을 넘기면 계획서 8절의 예외를 적용합니다. 9B 전체 base를 조기 제출로 쓰고, 27B는
validation과 FT 전체 추론만 담당하도록 줄입니다. 이 결정은 실행을 시작하기 전에 내려야 합니다.

In [28]:
steps = max(1, int(N_TRAIN_SAMPLES / (MICRO_BATCH * GRAD_ACCUM)))

plan = [
    ("base validation 추론", N_VAL_ITEMS * SEC_PER_ITEM / 3600.0),
    ("base 전체 test 추론", N_TEST_ITEMS * SEC_PER_ITEM / 3600.0),
    ("학습 1 epoch (%d step)" % steps, steps * SEC_PER_STEP / 3600.0),
    ("FT validation 추론", N_VAL_ITEMS * SEC_PER_ITEM / 3600.0),
    ("FT 전체 test 추론", N_TEST_ITEMS * SEC_PER_ITEM / 3600.0),
]
subtotal = sum(h for _, h in plan)
reserve = subtotal * 0.2
total = subtotal + reserve

print("%-28s %8s %8s" % ("작업", "시간(h)", "CU"))
print("-" * 48)
for name, hours in plan:
    print("%-28s %8.2f %8.1f" % (name, hours, hours * CU_RATE))
print("-" * 48)
print("%-28s %8.2f %8.1f" % ("소계", subtotal, subtotal * CU_RATE))
print("%-28s %8.2f %8.1f" % ("20% 예비", reserve, reserve * CU_RATE))
print("%-28s %8.2f %8.1f" % ("합계", total, total * CU_RATE))
print()
print("잔여 CU %.1f 중 %.1f를 사용하게 됩니다." % (CU_REMAINING, total * CU_RATE))

if total * CU_RATE > CU_REMAINING * 0.8:
    print()
    print("경고: 예산을 초과합니다. 계획서 8절의 예외를 적용할지 지금 결정해야 합니다.")
    print("  - 9B 전체 base 결과를 조기 제출로 사용합니다.")
    print("  - 27B는 validation과 FT 전체 추론만 담당합니다.")
else:
    print()
    print("PASS: 잔여 CU의 80퍼센트 안에 들어옵니다.")

budget = {"resources": RESOURCES, "config": CONFIG, "precision": PRECISION,
          "sec_per_item": SEC_PER_ITEM,
          "sec_per_microbatch": SEC_PER_MICRO,
          "grad_accum": GRAD_ACCUM,
          "sec_per_optimizer_step": SEC_PER_STEP,
          "sec_per_step": SEC_PER_STEP,  # 예전 이름. optimizer step 기준입니다.
          "peak_vram_gb": PEAK_VRAM, "plan_hours": dict(plan),
          "total_hours": round(total, 2), "total_cu": round(total * CU_RATE, 1)}
(RUN_ROOT_P / "budget.json").write_text(
    json.dumps(budget, ensure_ascii=False, indent=2), encoding="utf-8")
print()
print("예산을 저장했습니다:", RUN_ROOT_P / "budget.json")

작업                              시간(h)       CU
------------------------------------------------
base validation 추론               0.15      1.1
base 전체 test 추론                  1.98     14.8
학습 1 epoch (776 step)           13.70    102.8
FT validation 추론                 0.15      1.1
FT 전체 test 추론                    1.98     14.8
------------------------------------------------
소계                              17.95    134.7
20% 예비                           3.59     26.9
합계                              21.54    161.6

잔여 CU 600.0 중 161.6를 사용하게 됩니다.

PASS: 잔여 CU의 80퍼센트 안에 들어옵니다.

예산을 저장했습니다: /content/drive/MyDrive/vqa/runs/budget.json


In [29]:
print("===== Colab 준비 결과 공유 =====")
print("GPU         : %s / VRAM %.1fGB" % (RESOURCES.get("gpu_name"), RESOURCES.get("vram_gb", 0)))
print("RAM         : %s GB" % RESOURCES.get("ram_gb"))
print("로컬 디스크 : 여유 %.1fGB" % RESOURCES.get("local_disk_free_gb", 0))
print("Drive       : 여유 %.1fGB" % RESOURCES.get("drive_free_gb", 0))
print("잔여 CU     : %.1f (%.2f CU/h, %s)" % (CU_REMAINING, CU_RATE, CU_RATE_SOURCE))
print("config      : %s (%s)" % (CONFIG, PRECISION))
print("model       : %s @ %s" % (MODEL_ID, MODEL_REVISION))
print("가중치 캐시 : %s" % CACHE_DIR)
print("실측        : %.3f초/문항, %.2f초/optimizer step (= microbatch %d개)"
      % (SEC_PER_ITEM, SEC_PER_STEP, GRAD_ACCUM))
print("예산        : %.2f시간 / %.1f CU (20%% 예비 포함)" % (total, total * CU_RATE))
print("다음 단계   : 21_colab_27b_run.ipynb 에서 base validation부터 시작합니다.")
print("===============================")

===== Colab 준비 결과 공유 =====
GPU         : NVIDIA A100-SXM4-40GB / VRAM 40.0GB
RAM         : 83.5 GB
로컬 디스크 : 여유 188.3GB
Drive       : 여유 178.9GB
잔여 CU     : 600.0 (7.50 CU/h, 가정값)
config      : configs/colab_qwen35_27b_nf4.yaml (nf4)
model       : Qwen/Qwen3.5-27B @ main
가중치 캐시 : /content/drive/MyDrive/vqa/hf_cache
실측        : 1.061초/문항, 63.57초/optimizer step (= microbatch 8개)
예산        : 21.54시간 / 161.6 CU (20% 예비 포함)
다음 단계   : 21_colab_27b_run.ipynb 에서 base validation부터 시작합니다.
